# Octree Querying Using Python

In [2]:
import numpy as np

The purpose of this notebook is to create and test recursive functions like the octree spatial querying algorithm. Here we define a 3d space using a numpy array and put a number of objects within it at random. These objects will have bounds, which the octree will have to adequately navigate around.

To start off with, we input just the object centers in a 32 x 32 x 32 numpy cube.

In [10]:
grid_3d = np.zeros([32, 32, 32])
grid_3d.shape

(32, 32, 32)

To this space, we add 10 random "objects". These shall be point objects for the time being, but eventually we will add bounding boxes as well.

In [11]:
object_dict = {}

for i in range(1, 11):
    X = np.random.randint(0,32)
    Y = np.random.randint(0,32)
    Z = np.random.randint(0,32)

    object_dict[i] = (X, Y, Z)

    grid_3d[Z][Y][X] = i

np.count_nonzero(grid_3d)

np.int64(10)

Now we define our octree algorithms. First, we need to find the locations of these objects in space and assign them to the lowest bounds of our octree.

In [12]:
max_depth = 4

In [15]:
def make_octree(x_l, x_r, y_t, y_b, z_f, z_b, depth):

    octree = {}

    if depth == 0:
        x_l = int(x_l)
        x_r = int(x_r)
        y_t = int(y_t)
        y_b = int(y_b)
        z_f = int(z_f)
        z_b = int(z_b)
        octree['bounds'] = (x_l, x_r, y_t, y_b, z_f, z_b)

        lowest_bounds = grid_3d[z_f:z_b, y_b:y_t, x_l:x_r]
        octree['object_indices'] = lowest_bounds[lowest_bounds != 0]

        return octree

    octree['bounds'] = (x_l, x_r, y_t, y_b, z_f, z_b)
    octree[0] = make_octree(x_l, x_r - (x_r - x_l)/2, y_t - (y_t - y_b)/2, y_b, z_f, z_b - (z_b - z_f)/2, depth-1)
    octree[1] = make_octree(x_r - (x_r - x_l)/2, x_r, y_t - (y_t - y_b)/2, y_b, z_f, z_b - (z_b - z_f)/2, depth-1)
    octree[2] = make_octree(x_l, x_r - (x_r - x_l)/2, y_t, y_t - (y_t - y_b)/2, z_f, z_b - (z_b - z_f)/2, depth-1)
    octree[3] = make_octree(x_r - (x_r - x_l)/2, x_r, y_t, y_t - (y_t - y_b)/2, z_f, z_b - (z_b - z_f)/2, depth-1)
    octree[4] = make_octree(x_l, x_r - (x_r - x_l)/2, y_t - (y_t - y_b)/2, y_b, z_b - (z_b - z_f)/2, z_b, depth-1)
    octree[5] = make_octree(x_r - (x_r - x_l)/2, x_r, y_t - (y_t - y_b)/2, y_b, z_b - (z_b - z_f)/2, z_b, depth-1)
    octree[6] = make_octree(x_l, x_r - (x_r - x_l)/2, y_t, y_t - (y_t - y_b)/2, z_b - (z_b - z_f)/2, z_b, depth-1)
    octree[7] = make_octree(x_r - (x_r - x_l)/2, x_r, y_t, y_t - (y_t - y_b)/2, z_b - (z_b - z_f)/2, z_b, depth-1)

    return octree

Let's run a test to see what thr octree looks like.

In [16]:
test_ot = make_octree(0,32, 32, 0, 0, 32, max_depth)
test_ot

{'bounds': (0, 32, 32, 0, 0, 32),
 0: {'bounds': (0, 16.0, 16.0, 0, 0, 16.0),
  0: {'bounds': (0, 8.0, 8.0, 0, 0, 8.0),
   0: {'bounds': (0, 4.0, 4.0, 0, 0, 4.0),
    0: {'bounds': (0, 2, 2, 0, 0, 2),
     'object_indices': array([], dtype=float64)},
    1: {'bounds': (2, 4, 2, 0, 0, 2),
     'object_indices': array([], dtype=float64)},
    2: {'bounds': (0, 2, 4, 2, 0, 2),
     'object_indices': array([], dtype=float64)},
    3: {'bounds': (2, 4, 4, 2, 0, 2),
     'object_indices': array([], dtype=float64)},
    4: {'bounds': (0, 2, 2, 0, 2, 4),
     'object_indices': array([], dtype=float64)},
    5: {'bounds': (2, 4, 2, 0, 2, 4),
     'object_indices': array([], dtype=float64)},
    6: {'bounds': (0, 2, 4, 2, 2, 4),
     'object_indices': array([], dtype=float64)},
    7: {'bounds': (2, 4, 4, 2, 2, 4),
     'object_indices': array([], dtype=float64)}},
   1: {'bounds': (4.0, 8.0, 4.0, 0, 0, 4.0),
    0: {'bounds': (4, 6, 2, 0, 0, 2),
     'object_indices': array([], dtype=float64)},

Now lets create an intersection function.

In [17]:
def is_intersecting_3d(pos: tuple, coords: tuple, threshold=4):
    """
    Docstring for is_intersecting
    
    :param pos: Description
    :type pos: tuple
    :param input_tree: Description
    :param threshold: Description
    """
    camera_x, camera_y, camera_z = pos
    x_l, x_r, y_t, y_b, z_f, z_b = coords

    # Find where the camera is with reference to the box
    # find the X, Y and Z coordinate of the nearest side of the box
    closest_x = max(x_l, min(camera_x, x_r))
    closest_y = max(y_b, min(camera_y, y_t))
    closest_z = max(z_f, min(camera_z, z_b))

    # find the length of the line from camera center to box edge
    curr_dist = (closest_x - camera_x)**2 + (closest_y - camera_y)**2 + (closest_z - camera_z)**2
    
    if curr_dist < threshold**2:
        return True
    else:
        return False

We run a test to see if this function is working as expected. We should see a False here.

In [18]:
test_pos = (0,0,0)
test_coords = (30, 32, 32, 30, 30, 32)

is_intersecting_3d(test_pos, test_coords, threshold=5)

False

And we should see a True here.

In [19]:
test_pos = (28,28,28)
test_coords = (30, 32, 32, 30, 30, 32)

is_intersecting_3d(test_pos, test_coords, threshold=5)

True

Great. Now let's create our main search query function. This function will take in a camera position, run an intersection test at the various levels of the octree, and finally return the object indices of the octree levels which intersect with the search radius.

In [ ]:
def octree_search(ot, camera_pos, radius, depth=max_depth):
    pass